# Treino no Hard Hat Workers (Roboflow) até o ONNX com passaporte

**Decisão que o modelo automatiza:** em uma imagem de obra, existe cabeça **sem capacete**? Se sim, **ALERTA**.
Se a melhor candidata ficar na zona incerta, **VERIFICAR** (um humano olha). Caso contrário, **OK**.

Fluxo deste notebook:

1. baixa o dataset [Hard Hat Workers](https://universe.roboflow.com/joseph-nelson/hard-hat-workers) do Roboflow Universe;
2. treina um YOLO11n;
3. localiza o `best.pt` (época de maior *fitness* na validação);
4. exporta para ONNX e grava o **passaporte** (`lia.passaporte`) com classes em português, pré-processamento,
   regra de decisão e as métricas da época do best.

O arquivo final `capacete_best.onnx` roda no leitor web (`web/index.html`) sem nenhum arquivo extra.

**Antes de rodar:** Ambiente de execução → Alterar tipo → **GPU T4**. Cadastre a chave do Roboflow em
🔑 *Secrets* (barra lateral do Colab) com o nome `ROBOFLOW_API_KEY` e habilite o acesso do notebook.
A chave nunca aparece no código.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install -U ultralytics roboflow onnx onnxslim onnxruntime jsonschema

## 1. Código do passaporte

Clona só a pasta desta entrega do repositório. Se o repositório ainda não estiver no GitHub, envie a pasta
`passaporte/` manualmente para `/content/entrega-03-onnx/passaporte/`.

In [ ]:
import os, sys, subprocess

REPO = "https://github.com/m9tzin/lia1_2026_2.git"
PASTA = "Entregas - Matheus Marinho/entrega-03-onnx"
BASE = "/content/entrega-03-onnx"

if not os.path.isdir(f"{BASE}/passaporte"):
    subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none", "--sparse", REPO, "/content/repo"], check=True)
    subprocess.run(["git", "-C", "/content/repo", "sparse-checkout", "set", PASTA], check=True)
    os.symlink(f"/content/repo/{PASTA}", BASE)

sys.path.insert(0, BASE)
from passaporte.carimbar import ler
from passaporte.de_ultralytics import exportar, metricas_do_best
print("passaporte importado de", BASE)

## 2. Dataset do Roboflow

O projeto tem várias versões com recortes diferentes de classes (`raw_AllClasses`, `raw_HeadHelmetClasses`...).
Usamos a que tem só **cabeça** e **capacete**: a classe `person` não ajuda na decisão e desbalanceia o treino.
A célula procura essa versão pelo nome; se não achar, use `ROBOFLOW_VERSAO` manualmente
(confira o número na página do dataset).

In [ ]:
from roboflow import Roboflow

try:
    from google.colab import userdata
    CHAVE = userdata.get("ROBOFLOW_API_KEY")
except Exception:
    CHAVE = os.environ.get("ROBOFLOW_API_KEY")
assert CHAVE, "Cadastre ROBOFLOW_API_KEY nos Secrets do Colab (ou como variável de ambiente)."

ROBOFLOW_VERSAO = None  # ex.: 2, para forçar uma versão específica

projeto = Roboflow(api_key=CHAVE).workspace("joseph-nelson").project("hard-hat-workers")
versoes = projeto.versions()
for v in versoes:
    print(f"versão {v.version}: {v.name}")

if ROBOFLOW_VERSAO is None:
    escolhida = next((v for v in versoes if "headhelmet" in v.name.lower().replace("_", "")), versoes[0])
else:
    escolhida = projeto.version(ROBOFLOW_VERSAO)
print("\nusando:", escolhida.version, escolhida.name)

dataset = escolhida.download("yolov8", location="/content/hardhat")
DATA_YAML = f"{dataset.location}/data.yaml"

In [ ]:
import yaml
from pathlib import Path

cfg = yaml.safe_load(Path(DATA_YAML).read_text())
nomes = cfg["names"] if isinstance(cfg["names"], list) else [cfg["names"][i] for i in sorted(cfg["names"])]
print("classes do dataset:", nomes)
for parte in ("train", "valid", "test"):
    pasta = Path(dataset.location) / parte / "images"
    if pasta.exists():
        print(f"{parte}: {len(list(pasta.glob('*')))} imagens")

## 3. Treino

`patience=10` interrompe se o *fitness* de validação parar de melhorar por 10 épocas. Ao final, o Ultralytics
grava em `runs/detect/capacete/weights/`:

- `last.pt`: pesos da última época;
- `best.pt`: pesos da época de **maior fitness** na validação. Na versão 8.4 o fitness de detecção é o mAP50-95
  (versões antigas usavam `0.1·mAP50 + 0.9·mAP50-95`). É esse que vai para produção.

In [ ]:
from ultralytics import YOLO

modelo = YOLO("yolo11n.pt")
modelo.train(data=DATA_YAML, epochs=30, imgsz=640, patience=10, batch=32,
             project="/content/runs/detect", name="capacete", exist_ok=True)

BEST = Path(modelo.trainer.best)
print("best.pt:", BEST)
print("métricas da época do best:", metricas_do_best(BEST))

In [ ]:
from IPython.display import Image, display
pasta = BEST.parent.parent
for arquivo in ("results.png", "confusion_matrix_normalized.png"):
    if (pasta / arquivo).exists():
        display(Image(filename=str(pasta / arquivo), width=900))

## 4. Exportar o best.pt com passaporte

A tradução é feita por nome, não por posição, para não depender da ordem das classes na versão escolhida.
A regra de decisão fica gravada no próprio modelo: quem abrir o `.onnx` em qualquer leitor que siga o
passaporte aplica a mesma política.

In [ ]:
TRADUCAO = {"head": "sem capacete", "helmet": "capacete", "person": "pessoa"}
classes_pt = [TRADUCAO.get(n.lower(), n) for n in nomes]
print(dict(zip(nomes, classes_pt)))

caminho, passaporte = exportar(
    BEST,
    destino="/content/capacete_best.onnx",
    classes=classes_pt,
    nome="Capacete em obra (YOLO11n)",
    dataset=f"roboflow joseph-nelson/hard-hat-workers v{escolhida.version} ({escolhida.name})",
    confianca_minima=0.35,
    decisao={
        "alertar_se": ["sem capacete"],
        "limiar_alerta": 0.50,
        "zona_incerta": [0.30, 0.50],
        "mensagem": "Trabalhador sem capacete",
    },
)
print(caminho)

In [ ]:
# Prova de que tudo está dentro do arquivo: relê só o .onnx.
import json
print(json.dumps(ler("/content/capacete_best.onnx"), ensure_ascii=False, indent=2))

## 5. Conferência rápida e download

Roda o ONNX pelo Ultralytics em algumas imagens de teste. O leitor web deve mostrar as mesmas caixas.

In [ ]:
testes = sorted((Path(dataset.location) / "test" / "images").glob("*"))[:4]
onnx_modelo = YOLO("/content/capacete_best.onnx", task="detect")
for r in onnx_modelo.predict(testes, conf=0.35, verbose=False):
    print(Path(r.path).name, [(r.names[int(b.cls)], round(float(b.conf), 2)) for b in r.boxes])

In [ ]:
from google.colab import files
files.download("/content/capacete_best.onnx")